# Gemini 見出し生成テスト

この notebook は、Discord 送信・翻訳・Truth Social 取得を行わず、`headline_prompt` と投稿本文から Gemini の見出し生成だけを実APIでテストします。

- `.env` の `GEMINI_API_KEY` を使用します。
- `headline_prompt` の `{content}` に本文を差し込みます。
- アプリ側と同じ最低限の整形結果も表示します。

In [3]:
from pathlib import Path
import json
import os
import re

import requests

try:
    from dotenv import load_dotenv
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("python-dotenv が必要です: pip install -r requirements.txt") from exc


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "truth-social-monitor").exists() and (candidate / "headline_prompt").exists():
            return candidate
    raise RuntimeError("リポジトリルートが見つかりません。notebook を repo 内で開いてください。")


ROOT = find_repo_root()
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
HEADLINE_PROMPT = (ROOT / "headline_prompt").read_text(encoding="utf-8").strip()

if not GEMINI_API_KEY:
    raise RuntimeError(".env に GEMINI_API_KEY を設定してください。")
if "{content}" not in HEADLINE_PROMPT:
    raise RuntimeError("headline_prompt に {content} が必要です。")

print(f"ROOT: {ROOT}")
print(f"GEMINI_MODEL: {GEMINI_MODEL}")
print("headline_prompt loaded")

ROOT: C:\Users\sin18\code\truth-social-monitor
GEMINI_MODEL: gemini-3.5-flash
headline_prompt loaded


## 1件だけ試す

`POST_TEXT` を差し替えて実行してください。

In [4]:
POST_TEXT = """We had a tremendous meeting today on energy, jobs, and bringing manufacturing back to America. Big announcements are coming soon, and we will keep fighting for American workers and families."""

GENERATION_CONFIG = {
}

prompt = HEADLINE_PROMPT.replace("{content}", POST_TEXT)
print(prompt)

以下の文章は、ドナルド・トランプのTruth Socialアカウントからの投稿です。
この文章について、最も重要な情報を簡潔に伝えることができる、プロ向けのニュースの見出しを作成してください。
We had a tremendous meeting today on energy, jobs, and bringing manufacturing back to America. Big announcements are coming soon, and we will keep fighting for American workers and families.
ただし、以下のルールに従ってください。
- 見出しは日本語で作成してください
- 見出しは15〜35文字程度で簡潔にしてください
- 投稿で最も重要な主張、争点、対象、行動のうち少なくとも1つを必ず含めてください
- 途中で切れた表現にせず、日本語として完結した見出しにしてください
- 見出しは、投稿の内容にのみ基づいて作成し、あなたが判断した要素は入れないでください
- 前置きはなしで、見出しのみを出力してください


In [5]:
def call_gemini_headline(prompt: str) -> dict:
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"
    payload = {
        "contents": [
            {
                "parts": [
                    {"text": prompt}
                ]
            }
        ],
        "generationConfig": GENERATION_CONFIG,
    }
    response = requests.post(url, json=payload, headers={"Content-Type": "application/json"}, timeout=60)
    print("HTTP", response.status_code)
    response.raise_for_status()
    return response.json()


result = call_gemini_headline(prompt)
print(json.dumps(result, ensure_ascii=False, indent=2))

HTTP 200
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "エネルギーや雇用、製造業回帰で会議　近く重大発表へ",
            "thoughtSignature": "EuQhCuEhAQw51sdlPYri6rYX6iUy2aB3LLamuGN7mbKqKoxQlqivac9dkP6MGEKh+jAZRQFlm9FZ3zvL9bSqk9W4awcXFIw8vrV2zoajWXuACwUKRuCXCU9UOuswb/8Uu+zGFWeQWDy7EkqYoUDdo/mGEZfxjyfPGCMrKF5Ymit/zjwpow+A/Xdg+yAE/+14Rk4dm51SqTM/TpphGLWrHbDyPz3po9DzVKaxazcuGZb5+5FbPX053i8WGvPfAgUNPQaTUJSqNBWHmhaue1wnvdIagsNNM9X6Q1xTDQ1SFv1wobsrcp1IIZgDiAXli0AEI5nl9i14Tk3n35ThbdRztvcBKva3ZI/hO/g4KxPw2wxDgkTft5O+3rXcIJ2cp74HliS+yuT+EtYyPFwBB1++fHxcqaPyy20B/v+6OmPMtDhmRc/CTN/glKDmRjvq81S1XmQASdb3q0jhpF9PAVqhKeAvi40baWojsJVsKxJrPopd7mvjDVZ42C7H/Sbja0dc1qylEWAZdey7ob4pYbmjt3M5PgnKW23jTX/GJAwB3ZRHmIihOyMNjz0eMOq0lmM5hz6ZTFuM44L6WBmuiSsbkHGb5UQEI8PS6rKUS6nZvqZwoT3aqSvn7tUAvP4fEe7d+dQBe3xMJD6oxqTkbnqphiNa68tK6nVqjvQSVBUULpz4tMRFbZ9ZS9kk4yrkB9n36VrUdTon9s4/On34GrsKJWNql9P/oPaAQkdlMSJ6KBO3c8h96satMhUNAeqW1n7SnRFWw4l3PmjscleeLaG3EkaLhGDDPg8jhUT8Dvahl6lCmRf1qqKrurltsp9IRFzJM

In [ ]:
def normalize_single_line(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def sanitize_generated_headline(headline: str) -> str:
    headline = normalize_single_line(headline)
    headline = headline.strip("\"'「」『』“”‘’")
    headline = headline.rstrip("。")
    if not headline:
        return ""
    return headline[:77].rstrip() + "..." if len(headline) > 80 else headline


raw_headline = result["candidates"][0]["content"]["parts"][0].get("text", "")
app_title = sanitize_generated_headline(raw_headline)

print("RAW:", repr(raw_headline))
print("APP_TITLE:", app_title)
print("APP_TITLE_LENGTH:", len(app_title))

## 複数本文で試す

必要ならこのセルを実行してください。Gemini API を本文数ぶん呼びます。

In [ ]:
POSTS = [
    "The border must be secure. We need strong leadership, common sense, and policies that protect American families and communities.",
    "Inflation is hurting hardworking Americans. We are going to bring prices down, unleash American energy, and rebuild our economy.",
    "Small businesses are the backbone of our country. We will cut unnecessary regulations and make it easier to hire, build, and grow.",
]

for i, text in enumerate(POSTS, 1):
    prompt = HEADLINE_PROMPT.replace("{content}", text)
    result = call_gemini_headline(prompt)
    raw = result["candidates"][0]["content"]["parts"][0].get("text", "")
    title = sanitize_generated_headline(raw)
    print(f"{i}. {title}  ({len(title)} chars)")
    print(f"   raw={raw!r}")